# Conformal Triage — Fase 1+2 en Colab, **v2**

Cambios vs v1: PAD-UFES-20 ahora se baja del **ISIC Archive (colección 406, mirror oficial)**
con `isic-cli` — el link de Mendeley murió. Todo es reanudable: lo ya bajado/extraído se salta.

**Antes de correr:** `Entorno de ejecución → Cambiar tipo de entorno → GPU T4`. Después `Ejecutar todas`.
Tarda ~35–50 min (la extracción de ISIC 2019 trabaja callada por tramos: es normal, no la cortes).
Al final quedan ~61 MB en `conformal-triage/emb/` de tu Google Drive: la unica entrada que necesita el notebook 02.


In [ ]:
# 1) GPU + dependencias
!nvidia-smi -L
!pip -q install timm==0.9.16 open_clip_torch gdown isic-cli
!mkdir -p /content/data/isic2019 /content/data/hiba /content/data/pad /content/emb /content/ckpt


In [ ]:
%%bash
# 2) ISIC 2019 (9.1 GB, links oficiales del challenge) — se salta si ya esta
cd /content/data/isic2019
N=$(find . -name '*.jpg' 2>/dev/null | wc -l)
if [ "$N" -ge 25331 ]; then echo "ISIC 2019 ya esta ($N jpg), salto"; else
  wget -qc https://isic-archive.s3.amazonaws.com/challenges/2019/ISIC_2019_Training_Input.zip
  wget -qc https://isic-archive.s3.amazonaws.com/challenges/2019/ISIC_2019_Training_GroundTruth.csv
  unzip -qn ISIC_2019_Training_Input.zip
  echo "ISIC 2019: $(find . -name '*.jpg' | wc -l) jpg (esperado 25331)"
fi


In [ ]:
# 3) HIBA y PAD-UFES-20, ambos desde el ISIC Archive con isic-cli.
#    El ID de cada coleccion se busca por nombre (PAD es el mirror oficial del dataset de Mendeley).
import subprocess, re, glob, os

def isic_collection_id(needle):
    out = subprocess.run(['isic','collection','list'], capture_output=True, text=True).stdout
    line = next(l for l in out.splitlines() if needle.lower() in l.lower())
    nums = re.findall(r'\d+', line.split('│')[1] if '│' in line else line)
    return nums[0], line.strip()[:100]

def fetch(needle, dest, expected):
    imgs = glob.glob(f'{dest}/images/**/*.*', recursive=True)
    if len(imgs) >= expected:
        print(f'{dest}: ya hay {len(imgs)} archivos, salto'); return
    cid, line = isic_collection_id(needle)
    print(f'coleccion "{needle}" -> id {cid} | {line}')
    os.makedirs(f'{dest}/images', exist_ok=True)
    subprocess.run(['isic','metadata','download','--collections',cid], cwd=dest, check=False)
    subprocess.run(['isic','image','download','--collections',cid,f'{dest}/images/'], check=True)
    print(f'{dest}: {len(glob.glob(dest+chr(47)+"images"+chr(47)+"**"+chr(47)+"*.*", recursive=True))} archivos (esperado ~{expected})')

fetch('hospital italiano', '/content/data/hiba', 1616)
fetch('pad-ufes', '/content/data/pad', 2298)
# renombrar los csv de metadata que deja isic-cli, para llevarlos al Drive
for src, name in [('/content/data/hiba','hiba_isic_metadata.csv'), ('/content/data/pad','pad_isic_metadata.csv')]:
    for f in glob.glob(f'{src}/*.csv'):
        os.replace(f, f'/content/emb/{name}'); print('metadata ->', name)


In [ ]:
%%bash
# 4) PanDerm: repo + checkpoint ViT-L (Google Drive del README oficial) — se salta si ya esta
cd /content
[ -d PanDerm ] || git clone -q https://github.com/SiyuanYan1/PanDerm
CKPT=/content/ckpt/panderm_ll_data6_checkpoint-499.pth
if [ -s "$CKPT" ]; then echo 'checkpoint ya esta, salto'; else
  gdown 1SwEzaOlFV_gBKf2UzeowMC8z9UH7AQbE -O "$CKPT" || \
    echo 'Si gdown fallo por cuota: abri https://drive.google.com/file/d/1SwEzaOlFV_gBKf2UzeowMC8z9UH7AQbE/view , "Agregar acceso directo a mi unidad", monta tu Drive y copialo a /content/ckpt/'
fi
ls -lh /content/ckpt/


In [ ]:
# 5) Extraccion (PanDerm congelado, una pasada por imagen). Se salta lo ya extraido.
import sys, os, time, json, contextlib, io, numpy as np, pandas as pd, torch
from pathlib import Path
from torch.utils.data import Dataset, DataLoader
from PIL import Image, ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True
sys.path.insert(0, '/content/PanDerm/classification')
with contextlib.redirect_stdout(io.StringIO()):   # silencia el print gigante del modelo
    from models import get_encoder
    class A: pretrained_checkpoint = '/content/ckpt/panderm_ll_data6_checkpoint-499.pth'
    model, tfm = get_encoder(A(), model_name='PanDerm_Large_LP')
model.eval().cuda()
print('modelo cargado: ViT-L, embedding 1024')
EXTS = {'.jpg','.jpeg','.png','.bmp','.tif','.tiff'}

def extract(img_dir, batch=64, workers=2):
    paths = sorted(p for p in Path(img_dir).rglob('*') if p.suffix.lower() in EXTS)
    class DS(Dataset):
        def __len__(s): return len(paths)
        def __getitem__(s, i):
            try: img = Image.open(paths[i]).convert('RGB')
            except Exception: img = Image.new('RGB', (224,224))
            return tfm(img), paths[i].stem
    dl = DataLoader(DS(), batch_size=batch, num_workers=workers, pin_memory=True)
    F, ids, t0 = [], [], time.time()
    with torch.no_grad():
        for bi,(x,st) in enumerate(dl):
            with torch.autocast('cuda', dtype=torch.float16):
                f = model.forward_features(x.cuda(non_blocking=True), is_train=False)
            F.append(f.float().cpu().numpy()); ids.extend(st)
            if bi % 10 == 0: print(f'  {min((bi+1)*batch,len(paths))}/{len(paths)} ({(bi+1)*batch/max(time.time()-t0,1e-9):.0f} img/s)', flush=True)
    return np.array(ids), np.concatenate(F).astype(np.float16)

def check(n, esperado, nombre):
    print(('OK' if n == esperado else f'OJO: {nombre} tiene {n}, esperaba {esperado}'), '-', nombre, n)

if not os.path.exists('/content/emb/hiba_emb.npz'):
    ids, F = extract('/content/data/hiba/images'); check(len(ids), 1616, 'hiba')
    np.savez_compressed('/content/emb/hiba_emb.npz', ids=ids, features=F)
else: print('hiba ya extraido, salto')
if not os.path.exists('/content/emb/pad_emb.npz'):
    ids, F = extract('/content/data/pad/images'); check(len(ids), 2298, 'pad')
    np.savez_compressed('/content/emb/pad_emb.npz', ids=ids, features=F)
else: print('pad ya extraido, salto')
if not os.path.exists('/content/emb/isic2019_emb_part0.npz'):
    gt = pd.read_csv('/content/data/isic2019/ISIC_2019_Training_GroundTruth.csv')
    CLS = ['MEL','NV','BCC','AK','BKL','DF','VASC','SCC']
    lab = dict(zip(gt['image'], gt[CLS].values.argmax(1)))
    ids, F = extract('/content/data/isic2019'); check(len(ids), 25331, 'isic2019')
    L = np.array([lab.get(i, -1) for i in ids], dtype=np.int16)
    assert (L >= 0).all(), 'ids sin label en ISIC 2019'
    for k, sl in enumerate(np.array_split(np.arange(len(ids)), 3)):
        np.savez_compressed(f'/content/emb/isic2019_emb_part{k}.npz', ids=ids[sl], features=F[sl], labels=L[sl], classes=np.array(CLS))
else: print('isic2019 ya extraido, salto')
json.dump({'dim': 1024, 'ckpt': 'panderm_ll_data6_checkpoint-499.pth', 'dtype': 'float16',
           'pad_source': 'ISIC Archive collection 406 (mirror oficial de PAD-UFES-20)',
           'date': time.strftime('%Y-%m-%d %H:%M')},
          open('/content/emb/extract_meta.json','w'), indent=2)
print('extraccion completa')


In [ ]:
# 6) Guardar en tu Google Drive (~61 MB) y listar
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p '/content/drive/MyDrive/conformal-triage/emb'
!cp /content/emb/* '/content/drive/MyDrive/conformal-triage/emb/'
!ls -lh '/content/drive/MyDrive/conformal-triage/emb'
print('LISTO. Artefactos en Drive: conformal-triage/emb/ (entrada del notebook 02).')
